# Experiment 10: Deploying an AI Microservice with Streaming, Caching and Observability

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To build a production-style AI API microservice using FastAPI with streaming LLM responses, semantic caching using Redis, request logging and LangSmith tracing for observability, simulating a real-world AI backend deployment.

## 2. Objectives

- To expose a language model as a REST API with FastAPI and validate the requests.
- To stream the model's answer to the client token by token (Server-Sent Events).
- To reduce cost and latency with a semantic cache stored in Redis.
- To record every request in a structured log and read the log for analysis.
- To trace the model call with LangSmith.
- To measure the effect of the cache with a load test.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- Docker (to run Redis): `docker run -d --name lab-redis -p 6379:6379 redis:7-alpine`
- API key for the language model (Gemini through the OpenAI SDK); optional LangSmith API key
- Required Python libraries: `fastapi`, `uvicorn`, `httpx`, `redis`, `openai`, `sentence-transformers`, `langsmith`, `pandas`

## 4. Theory

### A. AI microservice

A model in a notebook can serve only one person. A **microservice** wraps it in an HTTP API so that many clients (web pages, mobile apps, other services) can use it. FastAPI is a Python framework that validates inputs with Pydantic models, supports `async` handlers, and documents the API automatically.

### B. Streaming

A model generates text token by token. With **Server-Sent Events (SSE)** the server sends each piece as soon as it exists, so the user sees the first words after a fraction of the total time.

### C. Semantic caching

An ordinary cache returns a stored answer only for an identical request. A **semantic cache** converts the question to an embedding and returns the stored answer of a question with a similar meaning ("What is the capital of France?" and "Which city is France's capital?"). Cache hits cost no model call: they are faster and free. The **similarity threshold** controls the risk: too low returns wrong answers for different questions, too high rarely hits. Redis stores the entries, with an expiry time (TTL) so that old answers disappear.

### D. Observability

| Tool | What it records |
|---|---|
| Request log | One JSON line per request: id, path, status, latency, cache hit. |
| Metrics | Counters such as hits and misses. |
| LangSmith trace | The inputs, outputs, time and token use of every model call, shown as a tree in a web dashboard. |

## 5. Algorithm / Procedure

1. Start Redis and install the libraries.
2. Write the FastAPI service (`app.py`): request model, semantic cache, `/chat`, `/chat/stream`, `/metrics` and a logging middleware.
3. Start the service and call it with a client.
4. Test the cache with repeated and paraphrased questions.
5. Test streaming and compare the time to the first token with the total time.
6. Read the request log.
7. Run a load test with the cache switched on and off.
8. Enable LangSmith tracing and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q fastapi uvicorn httpx redis openai sentence-transformers langsmith pandas

### Step 2: The semantic cache

Each cache entry is a Redis hash with the question, the answer and the question's embedding (384 numbers stored as bytes). `lookup` compares the new question's embedding with all stored ones and returns the best answer if its cosine similarity reaches the threshold. Keys expire after one hour.

In [2]:
%%writefile app.py
import os, json, time, uuid, asyncio
import numpy as np
import redis.asyncio as redis
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
app = FastAPI(title="AI microservice")
llm = AsyncOpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=3)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
db = redis.Redis(host="localhost", port=6379)
MODEL, THRESHOLD, TTL = "gemini-2.5-flash-lite", 0.85, 3600


class SemanticCache:
    async def lookup(self, question):
        q = embedder.encode(question, normalize_embeddings=True)
        ids = [i.decode() for i in await db.smembers("cache:ids")]
        if not ids:
            return None
        pipe = db.pipeline()
        for i in ids:
            pipe.hget(f"cache:{i}", "vec")
        vecs = await pipe.execute()
        pairs = [(float(q @ np.frombuffer(v, dtype="float32")), i) for i, v in zip(ids, vecs) if v is not None]
        if not pairs:
            return None
        score, best = max(pairs)
        if score < THRESHOLD:
            return None
        await db.hincrby("stats", "hits", 1)
        return (await db.hget(f"cache:{best}", "answer")).decode()

    async def store(self, question, answer):
        i = uuid.uuid4().hex
        vec = embedder.encode(question, normalize_embeddings=True).astype("float32").tobytes()
        await db.hset(f"cache:{i}", mapping={"question": question, "answer": answer, "vec": vec})
        await db.expire(f"cache:{i}", TTL)
        await db.sadd("cache:ids", i)


cache = SemanticCache()

Writing app.py


### Step 3: Request model, logging middleware and LangSmith tracing

The middleware wraps every request: it gives it an id, measures the time and appends one JSON line to `requests.jsonl`. The handler sets `request.state.cache_hit` so that the log also records whether the cache was used. If `LANGSMITH_TRACING=true` and a LangSmith key is set, the model call is wrapped with `@traceable` and appears in the LangSmith dashboard; otherwise the wrapper does nothing.

In [3]:
%%writefile -a app.py


class ChatRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=2000)
    use_cache: bool = True


@app.middleware("http")
async def log_requests(request: Request, call_next):
    request.state.cache_hit = None
    rid, t0 = uuid.uuid4().hex[:8], time.time()
    response = await call_next(request)
    ms = round((time.time() - t0) * 1000, 1)
    with open("requests.jsonl", "a") as f:
        f.write(json.dumps({"id": rid, "time": time.strftime("%H:%M:%S"), "path": request.url.path, "status": response.status_code,
                            "ms": ms, "cache_hit": request.state.cache_hit}) + "\n")
    response.headers["X-Request-ID"] = rid
    return response


async def call_llm(prompt):
    r = await llm.chat.completions.create(model=MODEL, max_tokens=250, temperature=0, messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or ""

if os.environ.get("LANGSMITH_TRACING") == "true" and os.environ.get("LANGSMITH_API_KEY"):
    from langsmith import traceable
    call_llm = traceable(name="call_llm", run_type="llm")(call_llm)

Appending to app.py


### Step 4: The endpoints

`/chat` checks the cache first and calls the model only on a miss. `/chat/stream` does the same but sends the answer as server-sent events: a cached answer is replayed in small pieces, a fresh answer is streamed from the model and stored once it is complete. `/metrics` returns the cache counters.

In [4]:
%%writefile -a app.py


@app.get("/health")
async def health():
    return {"status": "ok"}


@app.post("/chat")
async def chat(req: ChatRequest, request: Request):
    cached = await cache.lookup(req.prompt) if req.use_cache else None
    request.state.cache_hit = cached is not None
    if cached is None:
        cached = await call_llm(req.prompt)
        if req.use_cache:
            await cache.store(req.prompt, cached)
    return {"answer": cached, "cache_hit": request.state.cache_hit}


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, request: Request):
    cached = await cache.lookup(req.prompt) if req.use_cache else None
    request.state.cache_hit = cached is not None

    async def events():
        if cached is not None:
            for i in range(0, len(cached), 20):
                yield f"data: {json.dumps({'token': cached[i:i + 20]})}\n\n"
        else:
            parts = []
            stream = await llm.chat.completions.create(model=MODEL, max_tokens=250, temperature=0, stream=True,
                                                       messages=[{"role": "user", "content": req.prompt}])
            async for chunk in stream:
                piece = chunk.choices[0].delta.content if chunk.choices else ""
                if piece:
                    parts.append(piece)
                    yield f"data: {json.dumps({'token': piece})}\n\n"
            if req.use_cache:
                await cache.store(req.prompt, "".join(parts))
        yield "data: [DONE]\n\n"
    return StreamingResponse(events(), media_type="text/event-stream")


@app.get("/metrics")
async def metrics():
    stats = {k.decode(): int(v) for k, v in (await db.hgetall("stats")).items()}
    return {"cache_entries": await db.scard("cache:ids"), "cache_hits": stats.get("hits", 0)}

Appending to app.py


### Step 5: Start the service

The service runs as a separate `uvicorn` process, as it would on a server. The notebook then acts as an ordinary client. The Redis cache is emptied first so that the experiment starts clean.

In [5]:
import subprocess, sys, time, json, os, httpx, redis, pandas as pd
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 180)

redis.Redis().flushall()
if os.path.exists("requests.jsonl"):
    os.remove("requests.jsonl")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--port", "8000", "--log-level", "warning"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
URL = "http://127.0.0.1:8000"
for _ in range(120):                                  # wait until the model has loaded and the service answers
    try:
        if httpx.get(URL + "/health", timeout=1).status_code == 200:
            break
    except httpx.HTTPError:
        time.sleep(1)
print(httpx.get(URL + "/health").json())

{'status': 'ok'}


### Step 6: Semantic cache test

The first question must go to the model. The paraphrases should be answered from the cache, and an unrelated question should miss.

In [6]:
def ask(prompt, **kw):
    t = time.time()
    r = httpx.post(URL + "/chat", json={"prompt": prompt, **kw}, timeout=60).json()
    return {"Question": prompt, "Cache hit": r["cache_hit"], "Latency (ms)": round((time.time() - t) * 1000), "Answer": r["answer"][:60].replace("\n", " ")}

cache_test = pd.DataFrame([ask(q) for q in [
    "What is the capital of France?",
    "What is the capital of France?",
    "Which city is the capital of France?",
    "Tell me the capital city of France",
    "How does photosynthesis work?"]])
cache_test

,Question,Cache hit,Latency (ms),Answer
0,What is the capital of France?,False,2436,The capital of France is **Paris**.
1,What is the capital of France?,True,50,The capital of France is **Paris**.
2,Which city is the capital of France?,True,89,The capital of France is **Paris**.
3,Tell me the capital city of France,True,43,The capital of France is **Paris**.
4,How does photosynthesis work?,False,1858,Photosynthesis is a remarkable and essential process that pl


### Step 7: Streaming test

`httpx.stream` reads the response line by line as it arrives. We record the time of the first token and the total time for a long answer, once from the model and once from the cache.

In [7]:
def stream(prompt):
    t0, first, n = time.time(), None, 0
    with httpx.stream("POST", URL + "/chat/stream", json={"prompt": prompt}, timeout=60) as r:
        for line in r.iter_lines():
            if line.startswith("data: ") and line[6:] != "[DONE]":
                first = first or time.time() - t0
                n += 1
    return {"Prompt": prompt[:45], "First token (s)": round(first, 2), "Total (s)": round(time.time() - t0, 2), "Chunks": n}

long_q = "Explain in about 120 words how the internet works."
stream_test = pd.DataFrame([stream(long_q), stream(long_q)], index=["From the model", "From the cache"])
stream_test

,Prompt,First token (s),Total (s),Chunks
From the model,Explain in about 120 words how the internet w,0.94,1.54,4
From the cache,Explain in about 120 words how the internet w,0.04,0.05,30


### Step 8: Load test with and without the cache

24 requests (6 questions, each asked in 4 different wordings) are sent by 4 parallel clients. First the cache is bypassed with `use_cache=false`; then the cache is used, starting empty.

In [8]:
from concurrent.futures import ThreadPoolExecutor
groups = [["What is machine learning?", "Explain machine learning", "Can you tell me what machine learning is?", "Define machine learning"],
          ["How do vaccines work?", "Explain how vaccines work", "What is the working principle of vaccines?", "How does a vaccine protect the body?"],
          ["What is the speed of light?", "How fast does light travel?", "Tell me the speed of light", "What is the velocity of light?"],
          ["Why is the sky blue?", "Explain why the sky looks blue", "What makes the sky blue?", "Reason for the blue colour of the sky"],
          ["What is an API?", "Explain what an API is", "Define the term API", "What does API mean in software?"],
          ["How does a car engine work?", "Explain the working of a car engine", "What is the principle of a car engine?", "How do car engines run?"]]
prompts = [g[i] for i in range(4) for g in groups]      # same meaning questions are spread through the test

def load_test(use_cache):
    redis.Redis().flushall()
    t0 = time.time()
    with ThreadPoolExecutor(4) as pool:
        out = list(pool.map(lambda p: ask(p, use_cache=use_cache), prompts))
    d = pd.DataFrame(out)
    return {"Requests": len(d), "Model calls": int((~d["Cache hit"]).sum()), "Hit rate": round(d["Cache hit"].mean(), 2),
            "Total time (s)": round(time.time() - t0, 1), "Mean latency (ms)": round(d["Latency (ms)"].mean())}

load_table = pd.DataFrame({"Cache off": load_test(False), "Cache on": load_test(True)}).T.astype({"Requests": int, "Model calls": int, "Mean latency (ms)": int})
load_table

,Requests,Model calls,Hit rate,Total time (s),Mean latency (ms)
Cache off,24.0,24.0,0.0,11.1,1758.0
Cache on,24.0,12.0,0.5,6.2,992.0


### Step 9: Read the request log and the metrics

In [9]:
log = pd.read_json("requests.jsonl", lines=True)
print(len(log), "requests logged | distinct paths:", log.path.unique().tolist())
log.groupby("cache_hit")["ms"].agg(["count", "median", "max"]).round(0)

57 requests logged | distinct paths: ['/health', '/chat', '/chat/stream']


,count,median,max
cache_hit,,,
0.0,39,1808.0,2420.0
1.0,16,34.0,93.0


In [10]:
print(httpx.get(URL + "/metrics").json())
log.tail(5)

{'cache_entries': 12, 'cache_hits': 12}


,id,time,path,status,ms,cache_hit
52,e44aaa20,22:38:30,/chat,200,19.0,1.0
53,24815069,22:38:30,/chat,200,19.0,1.0
54,520fd14a,22:38:30,/chat,200,2182.5,0.0
55,949f944c,22:38:30,/chat,200,1934.8,0.0
56,9b5dc50a,22:38:30,/chat,200,2212.7,0.0


### Step 10: LangSmith tracing

To trace the model calls, set these variables in `.env` and restart the service: `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY=<your key>` and `LANGSMITH_PROJECT=ai-microservice-lab`. Every call of `call_llm` is then sent to LangSmith with its input, output, latency and token counts, and each request appears as a run in the project dashboard.

In [11]:
if os.environ.get("LANGSMITH_API_KEY") and os.environ.get("LANGSMITH_TRACING") == "true":
    print("LangSmith tracing enabled for project", os.environ.get("LANGSMITH_PROJECT", "default"))

LangSmith tracing enabled for project ai-microservice-lab


In [12]:
server.terminate(); server.wait()
print("service stopped")

service stopped


## 7. Experimental Observations

### Observation Table 1: Semantic cache test

In [13]:
cache_test.drop(columns="Answer")

,Question,Cache hit,Latency (ms)
0,What is the capital of France?,False,2436
1,What is the capital of France?,True,50
2,Which city is the capital of France?,True,89
3,Tell me the capital city of France,True,43
4,How does photosynthesis work?,False,1858


### Observation Table 2: Streaming against waiting

In [14]:
stream_test

,Prompt,First token (s),Total (s),Chunks
From the model,Explain in about 120 words how the internet w,0.94,1.54,4
From the cache,Explain in about 120 words how the internet w,0.04,0.05,30


### Observation Table 3: Load test (24 requests, 4 parallel clients)

In [15]:
load_table

,Requests,Model calls,Hit rate,Total time (s),Mean latency (ms)
Cache off,24.0,24.0,0.0,11.1,1758.0
Cache on,24.0,12.0,0.5,6.2,992.0


**Observations**

- The semantic cache worked as intended. The first question about France went to the model (2,436 ms). The identical question, and two paraphrases ("Which city is the capital of France?" and "Tell me the capital city of France"), were answered from Redis in 43 to 89 ms, which is roughly 27 to 57 times faster than the first call. The unrelated question about photosynthesis correctly missed the cache.
- Streaming reduced the waiting time. For a 120-word answer the first words arrived after 0.94 s while the complete answer needed 1.54 s. From the cache the first token came after 0.04 s and the whole answer after 0.05 s.
- The load test shows the saving: with the cache off all 24 requests called the model and took 11.1 s in total (mean latency 1,758 ms). With the cache on only 12 requests reached the model (hit rate 0.50), the total time fell to 6.2 s and the mean latency to 992 ms. The hit rate is not higher because the four wordings of a question were sent at the same time by different clients, so several of them arrived before the first answer was stored.
- The request log recorded every call with an id, path, status, latency and cache flag. Cache hits had a median latency of 34 ms, against 1,808 ms for misses, and `/metrics` reported 12 entries and 12 hits, matching the load test.
- LangSmith tracing is switched on by environment variables only; with the variables set, every `call_llm` run is visible in the project dashboard with its inputs, outputs, latency and token counts.
- The threshold of 0.85 accepts paraphrases, but a lower value would also accept different questions with similar words, so it must be tuned on real traffic.

## 8. Result

A FastAPI microservice was built around the Gemini model with request validation, server-sent-event streaming, a Redis semantic cache, structured request logging and LangSmith tracing. Paraphrased questions were answered from the cache in 43 to 89 ms instead of about 2 s. Streaming delivered the first words after 0.94 s for an answer that took 1.54 s to complete. In a load test of 24 requests the cache reduced the model calls from 24 to 12, the total time from 11.1 s to 6.2 s and the mean latency from 1,758 ms to 992 ms.

## 9. Conclusion

A language model becomes a usable service when it is wrapped in a validated, documented API. Streaming improves the perceived speed because the user reads the first words long before the answer is complete. A semantic cache in Redis removes repeated model calls, giving answers in milliseconds at no cost, but its similarity threshold must be chosen carefully to avoid returning the answer to a different question. Structured request logs, counters and LangSmith traces make the behaviour of the service visible, which is necessary to find slow requests, measure the cache hit rate and debug model calls in production.